In [ ]:
# -*- coding: utf-8 -*-
"""Image Caption->ViT,TransformerDecoder[PyTorch]"""

from google.colab import drive
drive.mount('/content/drive')

from google.colab import files
files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download adityajn105/flickr8k

!unzip /content/flickr8k.zip

!pip install googletrans==3.1.0a0
!pip install timm  # For Vision Transformer models

import pandas as pd
import numpy as np
from collections import Counter
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
from torch.autograd import Variable
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import math
import torch.nn.functional as F
import pickle
import gc
import random
from googletrans import Translator
import timm
from torchvision import transforms

translator = Translator()
pd.set_option('display.max_colwidth', None)

df = pd.read_csv("/content/captions.txt", sep=',')
print(len(df))
display(df.head(3))

def remove_single_char_word(word_list):
    lst = []
    for word in word_list:
        if len(word)>1:
            lst.append(word)
    return lst

df['cleaned_caption'] = df['caption'].apply(lambda caption : ['<start>'] + [word.lower() if word.isalpha() else '' for word in caption.split(" ")] + ['<end>'])
df['cleaned_caption']  = df['cleaned_caption'].apply(lambda x : remove_single_char_word(x))

df['seq_len'] = df['cleaned_caption'].apply(lambda x : len(x))
max_seq_len = df['seq_len'].max()
print(max_seq_len)

df.drop(['seq_len'], axis = 1, inplace = True)
df['cleaned_caption'] = df['cleaned_caption'].apply(lambda caption : caption + ['<pad>']*(max_seq_len-len(caption)) )

display(df.head(2))

word_list = df['cleaned_caption'].apply(lambda x : " ".join(x)).str.cat(sep = ' ').split(' ')
word_dict = Counter(word_list)
word_dict =  sorted(word_dict, key=word_dict.get, reverse=True)

print(len(word_dict))
print(word_dict[:5])

vocab_size = len(word_dict)
print(vocab_size)

index_to_word = {index: word for index, word in enumerate(word_dict)}
word_to_index = {word: index for index, word in enumerate(word_dict)}
print(len(index_to_word), len(word_to_index))

df['text_seq']  = df['cleaned_caption'].apply(lambda caption : [word_to_index[word] for word in caption] )

display(df.head(2))

df = df.sort_values(by = 'image')
train = df.iloc[:int(0.9*len(df))]
valid = df.iloc[int(0.9*len(df)):]

print(len(train), train['image'].nunique())
print(len(valid), valid['image'].nunique())

train_samples = len(train)
print(train_samples)

unq_train_imgs = train[['image']].drop_duplicates()
unq_valid_imgs = valid[['image']].drop_duplicates()
print(len(unq_train_imgs), len(unq_valid_imgs))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

class extractImageFeatureViTDataSet():
    def __init__(self, data, model_name='vit_base_patch16_224', pretrained=True):
        self.data = data
        self.model = timm.create_model(model_name, pretrained=pretrained, num_classes=0).to(device).eval() # num_classes=0 to get feature embeddings
        self.transform = transforms.Compose([
            transforms.Resize(self.model.default_cfg['input_size']),
            transforms.ToTensor(),
            transforms.Normalize(mean=self.model.default_cfg['mean'], std=self.model.default_cfg['std'])
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        image_name = self.data.iloc[idx]['image']
        img_loc = '/content/Images/'+str(image_name)
        img = Image.open(img_loc).convert("RGB")
        t_img = self.transform(img).unsqueeze(0).to(device) # Add batch dimension

        with torch.no_grad():
            features = self.model(t_img).flatten(1) # Flatten the output features

        return image_name, features

train_ImageDataset_ViT = extractImageFeatureViTDataSet(unq_train_imgs)
train_ImageDataloader_ViT = DataLoader(train_ImageDataset_ViT, batch_size = 1, shuffle=False)

valid_ImageDataset_ViT = extractImageFeatureViTDataSet(unq_valid_imgs)
valid_ImageDataloader_ViT = DataLoader(valid_ImageDataset_ViT, batch_size = 1, shuffle=False)

extract_imgFtr_ViT_train = {}
for image_name, features in tqdm(train_ImageDataloader_ViT):
    extract_imgFtr_ViT_train[image_name[0]] = features.cpu() # Store on CPU

a_file = open("./EncodedImageTrainViT.pkl", "wb")
pickle.dump(extract_imgFtr_ViT_train, a_file)
a_file.close()

extract_imgFtr_ViT_valid = {}
for image_name, features in tqdm(valid_ImageDataloader_ViT):
    extract_imgFtr_ViT_valid[image_name[0]] = features.cpu() # Store on CPU

a_file = open("./EncodedImageValidViT.pkl", "wb")
pickle.dump(extract_imgFtr_ViT_valid, a_file)
a_file.close()

class FlickerDataSetViT():
    def __init__(self, data, pkl_file):
        self.data = data
        self.encodedImgs = pd.read_pickle(pkl_file)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        caption_seq = self.data.iloc[idx]['text_seq']
        target_seq = caption_seq[1:]+[0]
        image_name = self.data.iloc[idx]['image']
        image_tensor = self.encodedImgs[image_name].to(device) # Load to device here

        return torch.tensor(caption_seq), torch.tensor(target_seq), image_tensor

train_dataset_vit = FlickerDataSetViT(train, 'EncodedImageTrainViT.pkl')
train_dataloader_vit = DataLoader(train_dataset_vit, batch_size = 32, shuffle=True)

valid_dataset_vit = FlickerDataSetViT(valid, 'EncodedImageValidViT.pkl')
valid_dataloader_vit = DataLoader(valid_dataset_vit, batch_size = 32, shuffle=True)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=max_seq_len):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)
        self.register_buffer('pe', pe)

    def forward(self, x):
        if self.pe.size(0) < x.size(0):
            self.pe = self.pe.repeat(x.size(0), 1, 1).to(device)
        self.pe = self.pe[:x.size(0), : , : ]
        x = x + self.pe
        return self.dropout(x)

class ImageCaptionModelViT(nn.Module):
    def __init__(self, n_head, n_decoder_layer, vocab_size, embedding_size, vit_feature_size):
        super(ImageCaptionModelViT, self).__init__()
        self.vit_feature_proj = nn.Linear(vit_feature_size, embedding_size) # Project ViT features to embedding size
        self.pos_encoder = PositionalEncoding(embedding_size, 0.1)
        self.TransformerDecoderLayer = nn.TransformerDecoderLayer(d_model =  embedding_size, nhead = n_head)
        self.TransformerDecoder = nn.TransformerDecoder(decoder_layer = self.TransformerDecoderLayer, num_layers = n_decoder_layer)
        self.embedding_size = embedding_size
        self.embedding = nn.Embedding(vocab_size , embedding_size)
        self.last_linear_layer = nn.Linear(embedding_size, vocab_size)
        self.init_weights()

    def init_weights(self):
        initrange = 0.1
        self.embedding.weight.data.uniform_(-initrange, initrange)
        self.last_linear_layer.bias.data.zero_()
        self.last_linear_layer.weight.data.uniform_(-initrange, initrange)
        self.vit_feature_proj.bias.data.zero_()
        self.vit_feature_proj.weight.data.uniform_(-initrange, initrange)

    def generate_Mask(self, size, decoder_inp):
        decoder_input_mask = (torch.triu(torch.ones(size, size)) == 1).transpose(0, 1)
        decoder_input_mask = decoder_input_mask.float().masked_fill(decoder_input_mask == 0, float('-inf')).masked_fill(decoder_input_mask == 1, float(0.0))
        decoder_input_pad_mask = decoder_inp.float().masked_fill(decoder_inp == 0, float(0.0)).masked_fill(decoder_inp > 0, float(1.0))
        decoder_input_pad_mask_bool = decoder_inp == 0
        return decoder_input_mask, decoder_input_pad_mask, decoder_input_pad_mask_bool

    def forward(self, encoded_image, decoder_inp):
        # Project ViT features
        encoded_image = self.vit_feature_proj(encoded_image).unsqueeze(0).permute(1, 0, 2) # Add seq_len dimension of 1

        decoder_inp_embed = self.embedding(decoder_inp)* math.sqrt(self.embedding_size)
        decoder_inp_embed = self.pos_encoder(decoder_inp_embed)
        decoder_inp_embed = decoder_inp_embed.permute(1,0,2)

        decoder_input_mask, decoder_input_pad_mask, decoder_input_pad_mask_bool = self.generate_Mask(decoder_inp.size(1), decoder_inp)
        decoder_input_mask = decoder_input_mask.to(device)
        decoder_input_pad_mask = decoder_input_pad_mask.to(device)
        decoder_input_pad_mask_bool = decoder_input_pad_mask_bool.to(device)

        decoder_output = self.TransformerDecoder(tgt = decoder_inp_embed, memory = encoded_image, tgt_mask = decoder_input_mask, tgt_key_padding_mask = decoder_input_pad_mask_bool)
        final_output = self.last_linear_layer(decoder_output)
        return final_output,  decoder_input_pad_mask

EPOCH = 30
embedding_size = 512
n_head = 8
n_decoder_layer = 4
# Get the feature size of the ViT model
vit_model_name = 'vit_base_patch16_224'
vit_feature_size = timm.create_model(vit_model_name, pretrained=True, num_classes=0).embed_dim

ictModel_vit = ImageCaptionModelViT(n_head, n_decoder_layer, vocab_size, embedding_size, vit_feature_size).to(device)
optimizer_vit = torch.optim.Adam(ictModel_vit.parameters(), lr = 0.00001)
scheduler_vit = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_vit, factor = 0.8, patience=2, verbose = True)
criterion_vit = torch.nn.CrossEntropyLoss(reduction='none')
min_val_loss_vit = np.float64('Inf')

for epoch in tqdm(range(EPOCH)):
    total_epoch_train_loss_vit = 0
    total_epoch_valid_loss_vit = 0
    total_train_words_vit = 0
    total_valid_words_vit = 0
    ictModel_vit.train()

    ### Train Loop
    for caption_seq, target_seq, image_embed in train_dataloader_vit:
        optimizer_vit.zero_grad()
        image_embed = image_embed.to(device)
        caption_seq = caption_seq.to(device)
        target_seq = target_seq.to(device)

        output, padding_mask = ictModel_vit.forward(image_embed, caption_seq)
        output = output.permute(1, 2, 0)

        loss = criterion_vit(output,target_seq)
        loss_masked = torch.mul(loss, padding_mask)
        final_batch_loss = torch.sum(loss_masked)/torch.sum(padding_mask)

        final_batch_loss.backward()
        optimizer_vit.step()
        total_epoch_train_loss_vit += torch.sum(loss_masked).detach().item()
        total_train_words_vit += torch.sum(padding_mask)

    total_epoch_train_loss_vit = total_epoch_train_loss_vit/total_train_words_vit

    ### Eval Loop
    ictModel_vit.eval()
    with torch.no_grad():
        for caption_seq, target_seq, image_embed in valid_dataloader_vit:
            image_embed = image_embed.to(device)
            caption_seq = caption_seq.to(device)
            target_seq = target_seq.to(device)

            output, padding_mask = ictModel_vit.forward(image_embed, caption_seq)
            output = output.permute(1, 2, 0)

            loss = criterion_vit(output,target_seq)
            loss_masked = torch.mul(loss, padding_mask)

            total_epoch_valid_loss_vit += torch.sum(loss_masked).detach().item()
            total_valid_words_vit += torch.sum(padding_mask)

    total_epoch_valid_loss_vit = total_epoch_valid_loss_vit/total_valid_words_vit

    print("Epoch -> ", epoch," Training Loss (ViT) -> ", total_epoch_train_loss_vit.item(), "Eval Loss (ViT) -> ", total_epoch_valid_loss_vit.item() )

    if min_val_loss_vit > total_epoch_valid_loss_vit:
        print("Writing ViT Model at epoch ", epoch)
        torch.save(ictModel_vit, './BestModelViT')
        min_val_loss_vit = total_epoch_valid_loss_vit

    scheduler_vit.step(total_epoch_valid_loss_vit.item())

model_vit = torch.load('./BestModelViT', weights_only=False)
start_token = word_to_index['<start>']
end_token = word_to_index['<end>']
pad_token = word_to_index['<pad>']
max_seq_len = 33
print(start_token, end_token, pad_token)

valid_img_embed_vit = pd.read_pickle('EncodedImageValidViT.pkl')

def generate_caption_vit(K, img_nm):
    img_loc = '/content/Images/'+str(img_nm)
    image = Image.open(img_loc).convert("RGB")
    plt.imshow(image)

    model_vit.eval()
    valid_img_df = valid[valid['image']==img_nm]
    print("Actual Caption : ")
    print(valid_img_df['caption'].tolist())
    img_embed = valid_img_embed_vit[img_nm].to(device)


    input_seq = [pad_token]*max_seq_len
    input_seq[0] = start_token

    input_seq = torch.tensor(input_seq).unsqueeze(0).to(device)
    predicted_sentence = []
    with torch.no_grad():
        for eval_iter in range(0, max_seq_len):

            output, padding_mask = model_vit.forward(img_embed, input_seq)
            output = output[eval_iter, 0, :]

            values = torch.topk(output, K).values.tolist()
            indices = torch.topk(output, K).indices.tolist()

            next_word_index = random.choices(indices, values, k = 1)[0]

            next_word = index_to_word[next_word_index]

            input_seq[:, eval_iter+1] = next_word_index


            if next_word == '<end>' :
                break

            predicted_sentence.append(next_word)
    print("\n")
    print("Predicted caption : ")
    print(" ".join(predicted_sentence+['.']))
    translated_caption = translator.translate(" ".join(predicted_sentence+['.']), dest='ar').text
    print("\n")
    print("Predicted caption (Arabic): ")
    print(translated_caption)

generate_caption_vit(1, unq_valid_imgs.iloc[600]['image'])

generate_caption_vit(2, unq_valid_imgs.iloc[50]['image'])

generate_caption_vit(1, unq_valid_imgs.iloc[100]['image'])

generate_caption_vit(2, unq_valid_imgs.iloc[500]['image'])